In [1]:
"""
CAPM Beta Algorithmic Trading Case
Rotman BMO Finance Research and Trading Lab, Uniersity of Toronto (C) All rights reserved.

Preamble:
-> Code will have a small start up period; however, trades should only be executed once forward market price is available,
hence there should not be any issue caused.

-> Code only runs effectively if the News articles are formatted as they are now. The only way to get the required new data is by parsing the text.
"""

import signal
import requests
from time import sleep
import pandas as pd
# import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
import copy


CAPM_vals = {}
expected_return = {}


# class that passes error message, ends the program
class ApiException(Exception):
    pass


# code that lets us shut down if CTRL C is pressed
def signal_handler(signum, frame):
    global shutdown
    signal.signal(signal.SIGINT, signal.SIG_DFL)
    shutdown = True


API_KEY = {"X-API-Key": "114514"}
shutdown = False
session = requests.Session()
session.headers.update(API_KEY)


# code that gets the current tick
def get_tick(session):
    resp = session.get("http://localhost:9999/v1/case")
    if resp.ok:
        case = resp.json()
        return case["tick"]
    raise ApiException("fail - cant get tick")


# code that parses the first and latest news instances for forward market predictions and the risk free rate
# Important: this code only works if the only '%' character is in front of the RISK FREE RATE and the onle '$' character is in front of the forward price suggestions
def get_news(session):
    news = session.get("http://localhost:9999/v1/news")
    if news.ok:
        newsbook = news.json()
        for i in range(len(newsbook[-1]["body"])):
            if newsbook[-1]["body"][i] == "%":
                CAPM_vals["%Rf"] = round(
                    float(newsbook[-1]["body"][i - 4 : i]) / 100, 4
                )
        latest_news = newsbook[0]
        if len(newsbook) > 1:
            for j in range(len(latest_news["body"]) - 1, 1, -1):
                while latest_news["body"][j] != "$":
                    j -= 1
            CAPM_vals["forward"] = float(latest_news["body"][j + 1 : -1])
        return CAPM_vals
    raise ApiException("timeout")


# gets all the price data for all securities
def pop_prices(session):
    price_act = session.get("http://localhost:9999/v1/securities")
    if price_act.ok:
        prices = price_act.json()
        return prices
    raise ApiException("fail - cant get securities")


# Buy or Sell function, put in your own parameters
def buy_or_sell(session, expected_return):
    for i in expected_return.keys():
        sec = session.get(f"http://localhost:9999/v1/securities?ticker=" + f"{i}").json()[0]
        max_er, min_er = analyze_er(expected_return)

        if expected_return[i] > 0:
            if sec["postion"] < 0:
                r = session.post(
                    "http://localhost:9999/v1/orders",
                    params={
                        "ticker": i,
                        "type": "LIMIT",
                        "quantity": "10000",
                        "action": "BUY",
                        "price": sec["bid"],
                    },
                )
            print(r.json())
        elif expected_return[i] < -0.01:
            session.post(
                "http://localhost:9999/v1/orders",
                params={
                    "ticker": i,
                    "type": "LIMIT",
                    "quantity": "10000",
                    "action": "SELL",
                    "price": sec["ask"],
                },
            )


def pop_real_bid_ask(session, ticker):
    s = session.get(f"http://localhost:9999/v1/securities/book?ticker={ticker}").json()
    bid, ask = 0, 0

    for open_order in s['bids']:
        if open_order['quantity'] > 10000:
            bid = open_order['price']
            break

    for open_order in s['asks']:
        if open_order['quantity'] > 10000:
            ask = open_order['price']
            break

    if bid == 0 or ask == 0:
        raise ApiException("No bid or ask found")

    return bid, ask


def analyze_er(expected_return):
    # get the largest positive expected return and smallest negative expected return
    max_er = max(expected_return, key=expected_return.get)
    min_er = min(expected_return, key=expected_return.get)

    return max_er, min_er


def linear_regression(ritm, alpha):
    ritm_clean = ritm.dropna(subset=["%Rm"])
    alpha_clean = alpha.dropna(subset=["%Ri"])

    x = ritm_clean["%Rm"].values.reshape(-1, 1)
    y = alpha_clean['%Ri']

    x = sm.add_constant(x)

    model = sm.OLS(y, x).fit()

    return model

total_camp = []
with requests.Session() as session:
    session.headers.update(API_KEY)
    ritm = pd.DataFrame(columns=["RITM", "BID", "ASK", "LAST", "%Rm"])
    alpha = pd.DataFrame(columns=["ALPHA", "BID", "ASK", "LAST", "%Ri", "%Rm"])
    gamma = pd.DataFrame(columns=["GAMMA", "BID", "ASK", "LAST", "%Ri", "%Rm"])
    theta = pd.DataFrame(columns=["THETA", "BID", "ASK", "LAST", "%Ri", "%Rm"])
    while get_tick(session) < 600 and not shutdown:
        # update the forward market price and rf rate
        get_news(session)

        ##update RITM bid-ask dataframe
        pdt_RITM = pd.DataFrame(pop_prices(session)[0])
        ritmp = pd.DataFrame(
            {
                "RITM": "",
                "BID": pdt_RITM["bid"],
                "ASK": pdt_RITM["ask"],
                "LAST": (pdt_RITM["bid"] + pdt_RITM["ask"]) / 2,
                "%Rm": "",
            }, index=[0]
        )
        if ritm["BID"].empty or ritmp["LAST"].iloc[0] != ritm["LAST"].iloc[0]:
            ritm = pd.concat([ritmp, ritm.loc[:]]).reset_index(drop=True)
            ritm["%Rm"] = (ritm["LAST"] / ritm["LAST"].shift(-1)) - 1
            if ritm.shape[0] >= 31:
                ritm = ritm.iloc[:30]

        # generate expected market return paramter
        if "forward" in CAPM_vals.keys():
            CAPM_vals["%RM"] = (CAPM_vals["forward"] - ritm["LAST"].iloc[0]) / ritm[
                "LAST"
            ].iloc[0]
        else:
            CAPM_vals["%RM"] = ""

        ##update ALPHA bid-ask dataframe
        pdt_ALPHA = pd.DataFrame(pop_prices(session)[1])

        alpha_bid, alpha_ask = pop_real_bid_ask(session, "ALPHA")
        alphap = pd.DataFrame(
            {
                "ALPHA": "",
                "BID": alpha_bid,
                "ASK": alpha_ask,
                "LAST": (alpha_bid + alpha_ask) / 2,
                "%Ri": "",
                "%Rm": "",
            }, index=[0]
        )
        if alpha["BID"].empty or alphap["LAST"].iloc[0] != alpha["LAST"].iloc[0]:
            alpha = pd.concat([alphap, alpha.loc[:]]).reset_index(drop=True)
            alpha["%Ri"] = (alpha["LAST"] / alpha["LAST"].shift(-1)) - 1
            alpha["%Rm"] = (ritm["LAST"] / ritm["LAST"].shift(-1)) - 1
            if alpha.shape[0] >= 31:
                alpha = alpha.iloc[:30]

        ##update GAMMA bid-ask dataframe
        pdt_GAMMA = pd.DataFrame(pop_prices(session)[2])

        gamma_bid, gamma_ask = pop_real_bid_ask(session, "GAMMA")
        gammap = pd.DataFrame(
            {
                "GAMMA": "",
                "BID": gamma_bid,
                "ASK": gamma_ask,
                "LAST": (gamma_bid + gamma_ask) / 2,
                "%Ri": "",
                "%Rm": "",
            }, index=[0]
        )
        if gamma["BID"].empty or gammap["LAST"].iloc[0] != gamma["LAST"].iloc[0]:
            gamma = pd.concat([gammap, gamma.loc[:]]).reset_index(drop=True)
            gamma["%Ri"] = (gamma["LAST"] / gamma["LAST"].shift(-1)) - 1
            gamma["%Rm"] = (ritm["LAST"] / ritm["LAST"].shift(-1)) - 1
            if gamma.shape[0] >= 31:
                gamma = gamma.iloc[:30]

        ##update THETA bid-ask dataframe
        pdt_THETA = pd.DataFrame(pop_prices(session)[3])

        theta_bid, theta_ask = pop_real_bid_ask(session, "THETA")
        thetap = pd.DataFrame(
            {
                "THETA": "",
                "BID": theta_bid,
                "ASK": theta_ask,
                "LAST": (theta_bid + theta_ask) / 2,
                "%Ri": "",
                "%Rm": "",
            }, index=[0]
        )
        if theta["BID"].empty or thetap["LAST"].iloc[0] != theta["LAST"].iloc[0]:
            theta = pd.concat([thetap, theta.loc[:]]).reset_index(drop=True)
            theta["%Ri"] = (theta["LAST"] / theta["LAST"].shift(-1)) - 1
            theta["%Rm"] = (ritm["LAST"] / ritm["LAST"].shift(-1)) - 1
            if theta.shape[0] >= 31:
                theta = theta.iloc[:30]

        beta_alpha = (alpha["%Ri"].cov(ritm["%Rm"])) / (ritm["%Rm"].var())
        beta_gamma = (gamma["%Ri"].cov(ritm["%Rm"])) / (ritm["%Rm"].var())
        beta_theta = (theta["%Ri"].cov(ritm["%Rm"])) / (ritm["%Rm"].var())


        CAPM_vals["Beta - ALPHA"] = beta_alpha
        CAPM_vals["Beta - GAMMA"] = beta_gamma
        CAPM_vals["Beta - THETA"] = beta_theta

        # record camp, index is the tick
        total_camp.append(CAPM_vals.copy())
        print(total_camp)
        print(len(total_camp))

        if CAPM_vals["%RM"] != "":
            er_alpha = CAPM_vals["%Rf"] + CAPM_vals["Beta - ALPHA"] * (
                CAPM_vals["%RM"] - CAPM_vals["%Rf"]
            )
            er_gamma = CAPM_vals["%Rf"] + CAPM_vals["Beta - GAMMA"] * (
                CAPM_vals["%RM"] - CAPM_vals["%Rf"]
            )
            er_theta = CAPM_vals["%Rf"] + CAPM_vals["Beta - THETA"] * (
                CAPM_vals["%RM"] - CAPM_vals["%Rf"]
            )
        else:
            er_alpha = "Wait for market forward price"
            er_gamma = "Wait for market forward price"
            er_theta = "Wait for market forward price"

        expected_return["ALPHA"] = er_alpha
        expected_return["GAMMA"] = er_gamma
        expected_return["THETA"] = er_theta

        # print statement (print, expected_return function, any of the tickers, or CAPM_vals dictionary)
        print(expected_return)
        # if len(ritm) >= 30 and len(alpha) >= 30:
        #     model_a = linear_regression(ritm, theta)
        #     print("The coefficient of the linear regression model is: ", model_a.summary())


            # coef_g, intercept_g = linear_regression(ritm, gamma)
            # print("The coefficient of the linear regression model is: ", coef_g)
            # print("The intercept of the linear regression model is: ", intercept_g)
            #
            # coef_t, intercept_t = linear_regression(ritm, theta)
            # print("The coefficient of the linear regression model is: ", coef_t)
            # print("The intercept of the linear regression model is: ", intercept_t)
            #
            # cov_ag= alpha["%Ri"].cov(gamma["%Ri"])
            # cov_at= alpha["%Ri"].cov(theta["%Ri"])
            # cov_gt= gamma["%Ri"].cov(theta["%Ri"])
            #
            # var_rm= ritm["%Rm"].var()
            #
            # diff_ag=cov_ag-coef_g*coef_a*var_rm
            # diff_at=cov_at-coef_t*coef_a*var_rm
            # diff_gt=cov_gt-coef_t*coef_g*var_rm
            #
            # print("The difference between the covariance and the product of the coefficients and the variance is: ", diff_ag)
            # print("The difference between the covariance and the product of the coefficients and the variance is: ", diff_at)
            # print("The difference between the covariance and the product of the coefficients and the variance is: ", diff_gt)

        sleep(1)



[{'%Rf': 0.0029, 'forward': 4412.18, '%RM': -0.005918732890986816, 'Beta - ALPHA': nan, 'Beta - GAMMA': nan, 'Beta - THETA': nan}]
1
{'ALPHA': nan, 'GAMMA': nan, 'THETA': nan}


C:\Users\ytwei3\PycharmProjects\scratchpad\venv\lib\site-packages\pandas\core\nanops.py:1406: RuntimeWarning: Degrees of freedom <= 0 for slice
  return np.cov(a, b, ddof=ddof)[0, 1]
C:\Users\ytwei3\PycharmProjects\scratchpad\venv\lib\site-packages\numpy\lib\function_base.py:2480: RuntimeWarning: divide by zero encountered in true_divide
  c *= np.true_divide(1, fact)


[{'%Rf': 0.0029, 'forward': 4412.18, '%RM': -0.005918732890986816, 'Beta - ALPHA': nan, 'Beta - GAMMA': nan, 'Beta - THETA': nan}, {'%Rf': 0.0029, 'forward': 4412.18, '%RM': 0.0028023682625543883, 'Beta - ALPHA': nan, 'Beta - GAMMA': nan, 'Beta - THETA': nan}]
2
{'ALPHA': nan, 'GAMMA': nan, 'THETA': nan}
[{'%Rf': 0.0029, 'forward': 4412.18, '%RM': -0.005918732890986816, 'Beta - ALPHA': nan, 'Beta - GAMMA': nan, 'Beta - THETA': nan}, {'%Rf': 0.0029, 'forward': 4412.18, '%RM': 0.0028023682625543883, 'Beta - ALPHA': nan, 'Beta - GAMMA': nan, 'Beta - THETA': nan}, {'%Rf': 0.0029, 'forward': 4412.18, '%RM': 0.00022442923564838096, 'Beta - ALPHA': 1.8959478435671127, 'Beta - GAMMA': -2.917749386068962, 'Beta - THETA': 3.8139590476572343}]
3
{'ALPHA': -0.002172742620983663, 'GAMMA': 0.010706644955071, 'THETA': -0.0073045173243460396}
[{'%Rf': 0.0029, 'forward': 4412.18, '%RM': -0.005918732890986816, 'Beta - ALPHA': nan, 'Beta - GAMMA': nan, 'Beta - THETA': nan}, {'%Rf': 0.0029, 'forward': 441

In [ ]:
total_camp